In [ ]:
!pip install -q google-genai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 9.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
from google import genai
from google.genai import types
from google.colab import userdata
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEYS")
)
print("setup done")

setup done


In [4]:
from google.colab import files
uploaded = files.upload()
pdf_name = list(uploaded.keys())[0]
print("Uploaded:",pdf_name)

Saving UNHRC BVRIT MUN-BG-2026.pdf to UNHRC BVRIT MUN-BG-2026.pdf
Uploaded: UNHRC BVRIT MUN-BG-2026.pdf


In [5]:
from pypdf import PdfReader
reader = PdfReader(pdf_name)
print("Number of Pages:", len(reader.pages))
text = ""
for page in reader.pages:
  text += (page.extract_text() or "") + "\n"
print("Total characters:", len(text))
print()
print(text[:500])

Number of Pages: 9
Total characters: 11232


UNITED NATIONS HUMAN RIGHTS COUNCIL (UNHRC)
BVRITMUN 2026
Agenda: Addressing challenges in the protection of human rights in 
Bangladesh, with a focus on vulnerable populations.
Greetings, Esteemed Delegates,
 
Welcome to the United Nations Human Rights Council at BVRITMUN 2026. 
The agenda before you is both relevant and complex . Human rights
discussions demand precision, diplomatic restraint, intellectual effort and
integrity. They require delegates to differentiate between political rhetori


In [6]:
def chunk_text(text, chunk_size = 800, overlap = 150):
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunks.append(text[start:end])
    start = end - overlap
  return chunks

chunks = chunk_text(text)
print("Number of chunks:", len(chunks))
print()
print(chunks[0])
print()
print("Last chunk length:", len(chunks[-1]))

Number of chunks: 18


UNITED NATIONS HUMAN RIGHTS COUNCIL (UNHRC)
BVRITMUN 2026
Agenda: Addressing challenges in the protection of human rights in 
Bangladesh, with a focus on vulnerable populations.
Greetings, Esteemed Delegates,
 
Welcome to the United Nations Human Rights Council at BVRITMUN 2026. 
The agenda before you is both relevant and complex . Human rights
discussions demand precision, diplomatic restraint, intellectual effort and
integrity. They require delegates to differentiate between political rhetoric 
and binding legal obligation.
 
Delegates are expected to present well ‑ researched arguments grounded in 
human rights, treaty law, UN mechanisms, and accurate foreign policy
representation. You are encouraged to refer to relevant Universal Periodic
Review documents, treaty body observations, an

Last chunk length: 182


In [8]:
EMB_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 768
def embed_text(t):
  result = client.models.embed_content(
      model=EMB_MODEL,
      contents=t,
      config=types.EmbedContentConfig(
          output_dimensionality=EMBEDDING_DIMENSION
      )
  )
  return np.array(result.embeddings[0].values)


In [9]:
chunk_embeddings = []
for i, ch in enumerate(chunks):
  chunk_embeddings.append(embed_text(ch))
  print(f"Embedded chunk{i + 1}/{len(chunks)}")
chunk_embeddings = np.array(chunk_embeddings)
chunk_embeddings = chunk_embeddings / np.maximum(
    np.linalg.norm(chunk_embeddings, axis=1, keepdims=True),
    1e-12
)
print("\nStored:", len(chunks), "chunks")
print("Embedding shape:", chunk_embeddings.shape)

Embedded chunk1/18
Embedded chunk2/18
Embedded chunk3/18
Embedded chunk4/18
Embedded chunk5/18
Embedded chunk6/18
Embedded chunk7/18
Embedded chunk8/18
Embedded chunk9/18
Embedded chunk10/18
Embedded chunk11/18
Embedded chunk12/18
Embedded chunk13/18
Embedded chunk14/18
Embedded chunk15/18
Embedded chunk16/18
Embedded chunk17/18
Embedded chunk18/18

Stored: 18 chunks
Embedding shape: (18, 768)


In [10]:
def retrieve(question, k=3):
  q_emb = embed_text(question)
  q_emb = q_emb / np.maximum(np.linalg.norm(q_emb), 1e-12)
  scores = chunk_embeddings @ q_emb
  top_idx = np.argsort(scores)[::-1][:k]
  return [
      (int(i), chunks[i], float(scores[i]))
      for i in top_idx
  ]

In [11]:
results = retrieve(
    "How many member states are in UNHRC?"
)
for chunk_id, chunk, score in results:
  print(f"Chunk {chunk_id + 1} | Score: {score:.4f}")
  print(chunk)
  print("-" * 60)

Chunk 3 | Score: 0.7707
roduction to UNHRC
The United Nations Human Rights Council (UNHRC) was established in 2006
through UN General Assembly Resolution 60/251, where it replaced its
predecessor, i.e. the Commission on Human Rights. It is composed of 47
Member States elected by the General Assembly on the basis of “equitable 
geographic distribution” and a 3 year term on a rotational basis.
 
3. Mandate of the Committee
The UNHRC functions as a subsidiary organ of the UN General Assembly and
reports annually to it. Its mandate includes promoting universal respect for
human rights, preventing violations, responding to emergencies, and 
integrating human rights considerations throughout the UN system.
 
The Council cannot impose sanctions or authorize force. Instead, it exercises 
influence through recommendations
------------------------------------------------------------
Chunk 5 | Score: 0.6861
specific human rights 
issues through resolutions, urgent debates, interactive dialogues, 

In [12]:
MODEL = "gemini-3.1-flash-lite"
ANSWER_CONFIG = types.GenerateContentConfig(
    system_instruction=(
        "You answer questions using only the context provided."
        "If the answer is not supported by the context, reply exactly: "
        "'I could not find the answer in the provided context.'"
        "Never guess the answer. Cite the supporting chunk ID."
    ),
    temperature=0,
    max_output_tokens=500
)

In [25]:
def ask(question, k=3):
  top = retrieve(question, k)
  context = "\n\n".join(
      f"[Chunk {chunk_id + 1}]\n{chunk}"
      for chunk_id, chunk, score in top
  )
  prompt = f"""
CONTEXT FROM THE DOCUMENT:
{context}
QUESTION: {question}
Answer using omly the context above.
Cite the supporting chunk ID(s).
If the answer is not present in the context, reply exactly:
"I could not find this in the document."
"""
  response = client.models.generate_content(
      model=MODEL,
      contents=prompt,
      config=ANSWER_CONFIG
  )
  print("QUESTION:", question)
  print("\nANSWER:\n", response.text)
  print("\nRETRIEVED CHUNKS:")
  for chunk_id, chunk, score in top:
    print(f"\nChunk {chunk_id + 1} | Score: {score:.4f}")
    print(chunk)
  return response.text

In [21]:
ask("What is the Agenda of BVRITMUN 2026?")

QUESTION: What is the Agenda of BVRITMUN 2026?

ANSWER:
 The agenda of BVRITMUN 2026 is: Addressing challenges in the protection of human rights in Bangladesh, with a focus on vulnerable populations.

Supporting chunk ID: [Chunk 1]

RETRIEVED CHUNKS:

Chunk 1 | Score: 0.7693

UNITED NATIONS HUMAN RIGHTS COUNCIL (UNHRC)
BVRITMUN 2026
Agenda: Addressing challenges in the protection of human rights in 
Bangladesh, with a focus on vulnerable populations.
Greetings, Esteemed Delegates,
 
Welcome to the United Nations Human Rights Council at BVRITMUN 2026. 
The agenda before you is both relevant and complex . Human rights
discussions demand precision, diplomatic restraint, intellectual effort and
integrity. They require delegates to differentiate between political rhetoric 
and binding legal obligation.
 
Delegates are expected to present well ‑ researched arguments grounded in 
human rights, treaty law, UN mechanisms, and accurate foreign policy
representation. You are encouraged to refer t

'The agenda of BVRITMUN 2026 is: Addressing challenges in the protection of human rights in Bangladesh, with a focus on vulnerable populations.\n\nSupporting chunk ID: [Chunk 1]'

In [22]:

test_questions = [
    "In which year was the UNHRC established, and under which UN General Assembly resolution?",
    "How many member states does the UNHRC have, and what is their term length?",
    "What human rights concerns were raised during the July 2024 student protests in Bangladesh?",
    "What exact amount in US dollars was requested for the Rohingya humanitarian response?",
    "What percentage of votes did each political party receive in the January 2025 parliamentary elections?",
    "What is the exact name and section number of the Bangladeshi law governing the minimum age of marriage?"
]

for i, question in enumerate(test_questions, start=1):
    print("\n", "=" * 70)
    print(f"TEST {i}")
    ask(question)



TEST 1
QUESTION: In which year was the UNHRC established, and under which UN General Assembly resolution?

ANSWER:
 The UNHRC was established in 2006 through UN General Assembly Resolution 60/251.

[Chunk 3]

RETRIEVED CHUNKS:

Chunk 3 | Score: 0.8100
roduction to UNHRC
The United Nations Human Rights Council (UNHRC) was established in 2006
through UN General Assembly Resolution 60/251, where it replaced its
predecessor, i.e. the Commission on Human Rights. It is composed of 47
Member States elected by the General Assembly on the basis of “equitable 
geographic distribution” and a 3 year term on a rotational basis.
 
3. Mandate of the Committee
The UNHRC functions as a subsidiary organ of the UN General Assembly and
reports annually to it. Its mandate includes promoting universal respect for
human rights, preventing violations, responding to emergencies, and 
integrating human rights considerations throughout the UN system.
 
The Council cannot impose sanctions or authorize force. Ins

In [23]:

large_chunks = chunk_text(
    text,
    chunk_size=3000,
    overlap=150
)

print("Original chunks:", len(chunks))
print("Large chunks:", len(large_chunks))

large_embeddings = []

for i, ch in enumerate(large_chunks):
    large_embeddings.append(embed_text(ch))
    print(f"Embedded large chunk {i + 1}/{len(large_chunks)}")

large_embeddings = np.array(large_embeddings)

large_embeddings = large_embeddings / np.maximum(
    np.linalg.norm(large_embeddings, axis=1, keepdims=True),
    1e-12
)

print("Large embedding shape:", large_embeddings.shape)


Original chunks: 18
Large chunks: 4
Embedded large chunk 1/4
Embedded large chunk 2/4
Embedded large chunk 3/4
Embedded large chunk 4/4
Large embedding shape: (4, 768)


In [26]:
original_chunks = chunks
original_embeddings = chunk_embeddings

try:
    chunks = large_chunks
    chunk_embeddings = large_embeddings

    for question in test_questions[:3]:
        print("\n" + "=" * 70)
        ask(question)
finally:
    chunks = original_chunks
    chunk_embeddings = original_embeddings


QUESTION: In which year was the UNHRC established, and under which UN General Assembly resolution?

ANSWER:
 The UNHRC was established in 2006 through UN General Assembly Resolution 60/251.

Supporting Chunk: [Chunk 1]

RETRIEVED CHUNKS:

Chunk 1 | Score: 0.6867

UNITED NATIONS HUMAN RIGHTS COUNCIL (UNHRC)
BVRITMUN 2026
Agenda: Addressing challenges in the protection of human rights in 
Bangladesh, with a focus on vulnerable populations.
Greetings, Esteemed Delegates,
 
Welcome to the United Nations Human Rights Council at BVRITMUN 2026. 
The agenda before you is both relevant and complex . Human rights
discussions demand precision, diplomatic restraint, intellectual effort and
integrity. They require delegates to differentiate between political rhetoric 
and binding legal obligation.
 
Delegates are expected to present well ‑ researched arguments grounded in 
human rights, treaty law, UN mechanisms, and accurate foreign policy
representation. You are encouraged to refer to relevant U